**Dataset note:** This notebook uses the official IBM geographic CSV when uploaded, otherwise the bundled 56-launch historical fallback. The fallback is not interchangeable with the original IBM geographic dataset. Distances are straight-line to selected illustrative points, not verified nearest infrastructure.


<p style="text-align:center">
    <a href="https://skills.network/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDS0321ENSkillsNetwork26802033-2022-01-01" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Hands-on Lab: Interactive Visual Analytics with Folium**


In [2]:
import micropip
await micropip.install("folium")

Estimated time needed: **40** minutes


The launch success rate may depend on many factors such as payload mass, orbit type, and so on. It may also depend on the location and proximities of a launch site, i.e., the initial position of rocket trajectories. Finding an optimal location for building a launch site certainly involves many factors and hopefully we could discover some of the factors by analyzing the existing launch site locations.


In the previous exploratory data analysis labs, you have visualized the SpaceX launch dataset using `matplotlib` and `seaborn` and discovered some preliminary correlations between the launch site and success rates. In this lab, you will be performing more interactive visual analytics using `Folium`.


## Objectives


This lab contains the following tasks:

*   **TASK 1:** Mark all launch sites on a map
*   **TASK 2:** Mark the success/failed launches for each site on the map
*   **TASK 3:** Calculate the distances between a launch site to its proximities

After completed the above tasks, you should be able to find some geographical patterns about launch sites.


Let's first import required Python packages for this lab:


In [3]:
# In JupyterLite use: import piplite; await piplite.install(['folium', 'pandas'])
# Standard Jupyter: install folium using %pip install folium if needed.
import pandas as pd
import folium


In [4]:
import folium
import pandas as pd

In [5]:
# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster
# Import folium MousePosition plugin
from folium.plugins import MousePosition
# Import folium DivIcon plugin
from folium.features import DivIcon

If you need to refresh your memory about folium, you may download and refer to this previous folium lab:


[Generating Maps with Python](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/labs/v4/DV0101EN-Exercise-Generating-Maps-in-Python.ipynb)


In [6]:
## Task 1: Mark all launch sites on a map


First, let's try to add each site's location on a map using site's latitude and longitude coordinates


The following dataset with the name `spacex_launch_geo.csv` is an augmented dataset with latitude and longitude added for each site.


In [7]:
# Offline-friendly data load. Upload spacex_launch_geo_fallback.csv alongside this notebook.
# For the official IBM analysis, replace the fallback with spacex_launch_geo.csv when available.
import pandas as pd
from pathlib import Path
if Path('spacex_launch_geo.csv').exists():
    spacex_df = pd.read_csv('spacex_launch_geo.csv')
    print('Loaded OFFICIAL IBM geographic CSV:',spacex_df.shape)
elif Path('spacex_launch_geo_fallback.csv').exists():
    spacex_df = pd.read_csv('spacex_launch_geo_fallback.csv')
    print('Loaded HISTORICAL FALLBACK sample (NOT the IBM geographic lab dataset):',spacex_df.shape)
else:
    raise FileNotFoundError('Upload spacex_launch_geo.csv or bundled spacex_launch_geo_fallback.csv')
spacex_df.head()


Loaded HISTORICAL FALLBACK sample (NOT the IBM geographic lab dataset): (56, 4)


,Launch Site,Lat,Long,class
0,CCAFS LC-40,28.561857,-80.577366,0
1,CCAFS LC-40,28.561857,-80.577366,0
2,CCAFS LC-40,28.561857,-80.577366,0
3,CCAFS LC-40,28.561857,-80.577366,0
4,CCAFS LC-40,28.561857,-80.577366,0


Now, you can take a look at what are the coordinates for each site.


In [8]:
# Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.561857,-80.577366
1,CCAFS SLC-40,28.561857,-80.577366
2,KSC LC-39A,28.608058,-80.603956
3,VAFB SLC-4E,34.632834,-120.610745


Above coordinates are just plain numbers that can not give you any intuitive insights about where are those launch sites. If you are very good at geography, you can interpret those numbers directly in your mind. If not, that's fine too. Let's visualize those locations by pinning them on a map.


We first need to create a folium `Map` object, with an initial center location to be NASA Johnson Space Center at Houston, Texas.


In [9]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

In [10]:
import micropip
await micropip.install("folium")

import folium
from folium.plugins import MarkerCluster, MousePosition
from folium.features import DivIcon

print("Folium loaded successfully!")

Folium loaded successfully!


We could use `folium.Circle` to add a highlighted circle area with a text label on a specific coordinate. For example,


In [11]:
# Create a blue circle at NASA Johnson Space Center's coordinate with a popup label showing its name
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
# Create a blue circle at NASA Johnson Space Center's coordinate with a icon showing its name
marker = folium.map.Marker(
    nasa_coordinate,
    # Create an icon as a text label
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

and you should find a small yellow circle near the city of Houston and you can zoom-in to see a larger circle.


Now, let's add a circle for each launch site in data frame `launch_sites`


*TODO:*  Create and add `folium.Circle` and `folium.Marker` for each launch site on the site map


An example of folium.Circle:


`folium.Circle(coordinate, radius=1000, color='#000000', fill=True).add_child(folium.Popup(...))`


An example of folium.Marker:


`folium.map.Marker(coordinate, icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0), html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'label', ))`


In [12]:
# TASK 1: circles and labels for every launch site
site_map = folium.Map(location=[31.3,-96.0], zoom_start=4)
for _, row in launch_sites_df.iterrows():
    coord=[float(row['Lat']),float(row['Long'])]
    name=str(row['Launch Site'])
    folium.Circle(coord, radius=1000, color='#175CD3',fill=True,fill_opacity=0.25,popup=name).add_to(site_map)
    folium.Marker(coord,popup=name,tooltip=name,icon=folium.Icon(color='blue',icon='info-sign')).add_to(site_map)
site_map.save('folium_task1_sites.html')
site_map


The generated map with marked launch sites should look similar to the following:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_markers.png">
</center>


Now, you can explore the map by zoom-in/out the marked areas
, and try to answer the following questions:

*   Are all launch sites in proximity to the Equator line?
*   Are all launch sites in very close proximity to the coast?

Also please try to explain your findings.


In [13]:
# Task 2: Mark the success/failed launches for each site on the map


Next, let's try to enhance the map by adding the launch outcomes for each site, and see which sites have high success rates.
Recall that data frame spacex_df has detailed launch records, and the `class` column indicates if this launch was successful or not


In [14]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class
46,CCAFS SLC-40,28.561857,-80.577366,1
47,CCAFS SLC-40,28.561857,-80.577366,0
48,VAFB SLC-4E,34.632834,-120.610745,0
49,CCAFS SLC-40,28.561857,-80.577366,0
50,VAFB SLC-4E,34.632834,-120.610745,0
51,CCAFS SLC-40,28.561857,-80.577366,0
52,CCAFS SLC-40,28.561857,-80.577366,1
53,KSC LC-39A,28.608058,-80.603956,1
54,VAFB SLC-4E,34.632834,-120.610745,0
55,CCAFS SLC-40,28.561857,-80.577366,0


Next, let's create markers for all launch records.
If a launch was successful `(class=1)`, then we use a green marker and if a launch was failed, we use a red marker `(class=0)`


Note that a launch only happens in one of the four launch sites, which means many launch records will have the exact same coordinate. Marker clusters can be a good way to simplify a map containing many markers having the same coordinate.


Let's first create a `MarkerCluster` object


In [15]:
marker_cluster = MarkerCluster()


*TODO:* Create a new column in `spacex_df` dataframe called `marker_color` to store the marker colors based on the `class` value


In [16]:
# TASK 2: assign color for each booster landing outcome
spacex_df['marker_color'] = spacex_df['class'].map({1:'green',0:'red'})
print(spacex_df['marker_color'].value_counts())


marker_color
red      32
green    24
Name: count, dtype: int64


*TODO:* For each launch result in `spacex_df` data frame, add a `folium.Marker` to `marker_cluster`


In [17]:
# TASK 2: distinct color-coded launch outcome markers in clusters
outcome_map = folium.Map(location=[31.3,-96.0],zoom_start=4)
marker_cluster=MarkerCluster().add_to(outcome_map)
for _,row in spacex_df.iterrows():
    folium.Marker([float(row['Lat']),float(row['Long'])],
       popup=f"{row['Launch Site']} | Landing {'success' if int(row['class'])==1 else 'failure'}",
       icon=folium.Icon(color=row['marker_color'])).add_to(marker_cluster)
outcome_map.save('folium_task2_outcomes.html')
outcome_map


Your updated map may look like the following screenshots:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster.png">
</center>


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster_zoomed.png">
</center>


From the color-labeled markers in marker clusters, you should be able to easily identify which launch sites have relatively high success rates.


In [18]:
# TASK 3: Calculate the distances between a launch site to its proximities


Next, we need to explore and analyze the proximities of launch sites.


Let's first add a `MousePosition` on the map to get coordinate for a mouse over a point on the map. As such, while you are exploring the map, you can easily find the coordinates of any points of interests (such as railway)


In [19]:
# TASK 3 map with mouse coordinates to select actual proximity targets
proximity_map = folium.Map(location=[28.5618571,-80.577366], zoom_start=12)
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
MousePosition(position='topright',separator=' Long: ',prefix='Lat:',lat_formatter=formatter,lng_formatter=formatter).add_to(proximity_map)
proximity_map


Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


In [20]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

*TODO:* Mark down a point on the closest coastline using MousePosition and calculate the distance between the coastline point and the launch site.


In [21]:
# TASK 3: example coastline point from IBM lab prompt; verify visually on the interactive map.
# These points are illustrative, NOT verified as nearest coastline or road/rail access points.
selected_site = launch_sites_df[launch_sites_df['Launch Site'].str.contains('CCAFS',case=False)].iloc[0]
launch_site_lat,launch_site_lon = float(selected_site['Lat']),float(selected_site['Long'])
coastline_lat,coastline_lon = 28.56367,-80.57163
site_coord=[launch_site_lat,launch_site_lon]
coast_coord=[coastline_lat,coastline_lon]
distance_coastline=calculate_distance(*site_coord,*coast_coord)
print(f"Straight-line distance to selected example coastline point: {distance_coastline:.3f} km")


Straight-line distance to selected example coastline point: 0.596 km


In [22]:
# TASK 3: label point and measured example distance
folium.Marker(site_coord,tooltip='Selected launch site',icon=folium.Icon(color='blue')).add_to(proximity_map)
folium.Marker(coast_coord,tooltip=f'Example coastline point: {distance_coastline:.2f} km (straight-line)',
    icon=folium.Icon(color='orange')).add_to(proximity_map)
proximity_map


*TODO:* Draw a `PolyLine` between a launch site to the selected coastline point


In [23]:
# TASK 3: show distance between site and selected coastline point
lines=folium.PolyLine(locations=[site_coord,coast_coord],color='#d35400',weight=3,
    tooltip=f'Example coastline point: {distance_coastline:.2f} km straight-line')
proximity_map.add_child(lines)
proximity_map.save('folium_task3_proximity.html')
proximity_map


Your updated map with distance line should look like the following screenshot:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_distance.png">
</center>


*TODO:* Similarly, you can draw a line betwee a launch site to its closest city, railway, highway, etc. You need to use `MousePosition` to find the their coordinates on the map first


A railway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/railway.png">
</center>


A highway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/highway.png">
</center>


A city map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/city.png">
</center>


In [24]:
# Additional proximity comparisons require measured, verified coordinates.
# On map, hover to obtain coordinates for a road, railway, or nearby city; enter them below.
# Do NOT describe any of these as the *closest* without independent verification.
proximity_targets = {
    # 'Road point': (28.57000, -80.58000),
    # 'Rail point': (28.58000, -80.60000),
    # 'City point': (28.40000, -80.60000),
}
for target,(lat,lon) in proximity_targets.items():
    km=calculate_distance(launch_site_lat,launch_site_lon,lat,lon)
    folium.Marker([lat,lon],tooltip=f'{target}: {km:.2f} km',icon=folium.Icon(color='purple')).add_to(proximity_map)
    folium.PolyLine([site_coord,[lat,lon]],tooltip=f'{km:.2f} km',weight=2).add_to(proximity_map)
    print(target,round(km,2),'km straight-line')
proximity_map.save('folium_task3_proximity.html')
proximity_map


In [25]:
# Additional optional analysis cell


In [26]:
# Additional optional analysis cell


After you plot distance lines to the proximities, you can answer the following questions easily:

*   Are launch sites in close proximity to railways?
*   Are launch sites in close proximity to highways?
*   Are launch sites in close proximity to coastline?
*   Do launch sites keep certain distance away from cities?

Also please try to explain your findings.


# Next Steps:

Now you have discovered many interesting insights related to the launch sites' location using folium, in a very interactive way. Next, you will need to build a dashboard using Ploty Dash on detailed launch records.


## Authors


[Pratiksha Verma](https://www.linkedin.com/in/pratiksha-verma-6487561b1/)


<!--## Change Log--!>


<!--| Date (YYYY-MM-DD) | Version | Changed By      | Change Description      |
| ----------------- | ------- | -------------   | ----------------------- |
| 2022-11-09        | 1.0     | Pratiksha Verma | Converted initial version to Jupyterlite|--!>


### <h3 align="center"> IBM Corporation 2022. All rights reserved. <h3/>
